# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu not defterinde LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi belgelerimiz hakkında sorular sormak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [35]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [36]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Neden RAG?

Bir LLM kendi başına, öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verileri geçmişten gelir ve en son verilerle güncellenmez.
- Sadece eğitim aldığı verileri bilir.

Bir LLM'yi kendi verilerimizle çalışması için kullanmak istiyoruz. İşte bu noktada RAG (Retrieval-Augmented Generation) devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçek doğruluğu artırmak için bir dil modelini belge alıcı ile birleştirir.
2. **İlgili dış belgeleri alır** (örneğin, bilgi tabanından) yanıtlar üretmeden önce.
3. **Dil modeli hem istemi hem de alınan bağlamı kullanarak** daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu meydan okumada, Avrupa Parlamentosu'ndan belgelerle çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini öğrenmek istediğinizi düşünün. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri EP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri karıştırmak istemezsiniz. O halde, hayatımızı kolaylaştırmak için RAG'ı kullanalım!

Bu, her zaman test etmek için yepyeni veriler alabileceğimiz için çalışmak üzere iyi verilerdir.

## 📘 Verileri alalım

1. [EP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna yönlendirecektir.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın sonuna kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir belgeyle başlayacağız, ancak daha sonrası için diğer birkaç günün aynısını şimdiden indirebilirsiniz.

Belgeye bir göz atın. Kaç sayfası var? Belge hakkında bir fikir edinmek için hızlıca belgede gezinin.

## 🔢 Belgeleri gömme

Belgeleri gömmek, tüm belgeleri veya belge parçalarını vektörlere çevirmek anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir gömme aracı (embedder) başlatalım ve deneyelim. LLM olarak Gemini kullandığımız için, Google'ın metin gömme araçlarında kalalım.

In [37]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Basit bir metin parçasını gömmek için gömme aracının `.embed_query()` metodunu deneyin.

In [38]:
# Embed a text like "What is the capital of France?" and save it to a variable `sample_embedding`

embedded = embeddings.embed_query("What is the capital of France?")

👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Nasıl görünüyor? Tipi nedir? Gömme boyutu nedir?

In [39]:
len(embedded)

3072

## 💾 PDF'den gerçek verilerimizi yükle

Artık bir gömmenin nasıl göründüğünü biliyoruz, gerçek verilerimizle çalışmanın zamanı geldi.

👉 [LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'yi nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [40]:
%pip install -qU langchain-community pypdf
from langchain_community.document_loaders import PyPDFLoader

file_path = 'data/CRE-10-2026-03-25_EN.PDF'

loader = PyPDFLoader(file_path)

Note: you may need to restart the kernel to use updated packages.


👉 `pages`'i keşfedin:
- Veri tipi nedir?
- Kaç sayfanız var?
- Bir sayfanın tipi nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam belgenin kaç karakteri var?
- Bir sayfanın `metadata`'sında neler var?

In [41]:
pages = loader.load_and_split()
len(pages)

167

In [42]:
type(pages)

list

In [43]:
type(pages[0])

langchain_core.documents.base.Document

In [44]:
pages[0].page_content

'2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nTUARASCÁIL FOCAL AR FHOCAL NA N-IMEACHTAÍ  DOBESEDNI ZAPISI RAZPRAV \nDOSLOVNO IZVJEŠĆE  SANATARKAT ISTUNTOSELOSTUKSET \nRESOCONTO INTEGRALE DELLE DISCUSSIONI  FULLSTÄNDIGT FÖRHANDLINGSREFERAT \n \n \nСряда - Miércoles - Středa - Onsdag - Mittwoch - Kolmapäev - Τετάρτη - Wednesday \nMercredi - Dé Céadaoin - Srijeda - Mercoledì - Trešdiena - Trečiadienis - Szerda - L-Erbgħa \nWoensdag - Środa - Quarta-feira - Miercuri - Streda - Sreda - 

In [45]:
sum(len(page.page_content) for page in pages)

512517

In [46]:
pages[0].metadata

{'producer': 'Aspose.Words for Java 24.2.0',
 'creator': 'Aspose.Words',
 'creationdate': '',
 'author': 'e-Parliament@europarl.europa.eu',
 'dmxml.render.id': '440386',
 'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef',
 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml',
 'source': 'data/CRE-10-2026-03-25_EN.PDF',
 'total_pages': 167,
 'page': 0,
 'page_label': '1'}

## ✂️ Verilerimizi böl

Tam belgemiz gömülmek için çok uzun. Metin gömme aracımız 2.048 tokena kadar giriş alabilir. Gemini modelleri için bu yaklaşık 8.196 karakterdir (token başına 4 karakter).

Bu yüzden belgemizi daha küçük parçalara bölmek istiyoruz.

Zaten çalışabileceğimiz bir dizi sayfamız var. Ama sayfa sonları biraz keyfi: genellikle cümlenin ortasında görünürler.

Ayrıca, sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaçırır. Tam metni biraz örtüşmeyle bölmek daha iyidir.

İlk olarak, PDF'yi tekrar yükleyeceğiz, bu sefer bölmeden.

In [47]:
loader = PyPDFLoader(file_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)

512683

In [63]:
len(pdf)

1

Artık tüm PDF'imizi tek bir belge olarak aldığımıza göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Yine, ["Özyinelemeli olarak bölme" konusundaki LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _belgelerimizi_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl böleceğinizi öğrenin.

2_000 karakter (bizim durumumuzda yaklaşık yarım sayfa) parçalara 400 örtüşmeyle bölün. İsterseniz diğer değerlerle deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod giriş olarak bir belge alır ve bölünmüş belgeler çıktısı verir.

In [65]:
!pip install -qU langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400, add_start_index=True)
all_splits = text_splitter.split_documents(pdf)
print(f"Split transcript into {len(all_splits)} sub-documents.")

Split transcript into 321 sub-documents.


In [66]:
all_splits

[Document(metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '440386', 'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef', 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-03-25_EN.PDF', 'total_pages': 167, 'start_index': 0}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nTUARASCÁIL FOCAL AR FHOCAL

👉 `all_splits`'i inceleyin:
- Veri tipi nedir?
- Kaç bölümünüz var?
- Bir bölümün tipi nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Şimdi toplamda kaç karakterimiz var?
- Bir bölümün `metadata`'sında neler var?

In [67]:
type(all_splits)

list

In [68]:
type(all_splits[0])

langchain_core.documents.base.Document

In [69]:
all_splits[0].page_content

'2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nTUARASCÁIL FOCAL AR FHOCAL NA N-IMEACHTAÍ  DOBESEDNI ZAPISI RAZPRAV \nDOSLOVNO IZVJEŠĆE  SANATARKAT ISTUNTOSELOSTUKSET \nRESOCONTO INTEGRALE DELLE DISCUSSIONI  FULLSTÄNDIGT FÖRHANDLINGSREFERAT \n \n \nСряда - Miércoles - Středa - Onsdag - Mittwoch - Kolmapäev - Τετάρτη - Wednesday \nMercredi - Dé Céadaoin - Srijeda - Mercoledì - Trešdiena - Trečiadienis - Szerda - L-Erbgħa \nWoensdag - Środa - Quarta-feira - Miercuri - Streda - Sreda - 

In [70]:
all_splits[0].metadata

{'producer': 'Aspose.Words for Java 24.2.0',
 'creator': 'Aspose.Words',
 'creationdate': '',
 'author': 'e-Parliament@europarl.europa.eu',
 'dmxml.render.id': '440386',
 'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef',
 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml',
 'source': 'data/CRE-10-2026-03-25_EN.PDF',
 'total_pages': 167,
 'start_index': 0}

## 🗄️ Her şeyi bir araya getir: belgelerimizi gömme ve vektör deposunda sakla

Elimizde şunlar var:
- Bir gömme aracı
- Veriyi yüklemek için bir yükleyici
- Belgemizi belgelere bölmek için bir metin bölücü

Neyi kaçırıyoruz?

Belgelerimizi gömebiliriz, ama onları bir yerde saklamak istiyoruz. İşte burada vektör deposu devreye girer: şunları saklamamıza olanak sağlar:
- belgeyi (parçayı),
- onun gömmesini,
- meta verilerini.

Sonraki adımda belgeleri verimli bir şekilde alabilecek olacağız.

👉 Bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

In [71]:
from langchain_core.vectorstores import InMemoryVectorStore
# YOUR CODE HERE

# Create an in-memory vector store using the embedder `embeddings` we created earlier

vector_store = InMemoryVectorStore(embeddings)

# Add the `all_splits` to the vector store and store the result in a variable called `document_ids`

document_ids = vector_store.add_documents(documents=all_splits)

In [74]:
document_ids

['f87c8222-a336-4356-9490-44de550dc985',
 'bc6a02e5-580a-40b4-8e8b-e1ffbf17d9c1',
 '6f19723f-fdb7-48c5-bd32-7d869e19e9ca',
 '303ceacf-7203-4ea3-8ede-5459c751349c',
 '992ce511-e72b-473d-8507-2df5c3b8b162',
 '3066a53b-0d8b-4fc9-9027-2993c23c15b2',
 'ed387348-c63d-4684-93cd-65af476eccd1',
 '95e1e60f-27d4-4e61-8722-490bc528599b',
 '1947eb98-3712-47e4-a073-783cc0787c7e',
 '3b974eb7-5b31-41fb-a209-3760e7fae646',
 '17517654-6e9b-4dc1-b0b0-af6edf75aca1',
 'b686ed47-b5fe-4eb3-bc5d-24c55bc1ad47',
 '58b4f05f-c94d-44fb-84dc-49ca2abc95d1',
 'a5d8296e-959d-4861-b1ab-d0fcb59eaf17',
 'ca12417e-0e89-41ea-95c0-1164667a2dde',
 '3adf6f1b-46bb-4784-9f2f-aab4267f5e81',
 '2863cb9a-3a21-40c5-9463-777549ae1bbc',
 '7c3f6129-1f11-4d0d-b6b2-e78c6e1cabe5',
 'c6f7de8e-9a8e-486f-910e-a0f189b02f7c',
 '03eb7f86-31e2-45e3-aaaf-b0b24c9cfa97',
 '35daf75d-db7e-4c90-a6b6-52e58b0b0821',
 '58c7a80b-1ce1-4d81-94e7-c3a33cc0bd15',
 'f3cf3868-cc93-4c14-979b-9c0652df43c5',
 'dd9d5ad0-887f-4038-bab8-46bf2cf5480c',
 '773ba6d4-0178-

In [72]:
# Have a look at the first 3 document IDs

print(document_ids[:3])

['f87c8222-a336-4356-9490-44de550dc985', 'bc6a02e5-580a-40b4-8e8b-e1ffbf17d9c1', '6f19723f-fdb7-48c5-bd32-7d869e19e9ca']


In [73]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.

vector_store.get_by_ids(document_ids[:3])

[Document(id='f87c8222-a336-4356-9490-44de550dc985', metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '440386', 'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef', 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-03-25_EN.PDF', 'total_pages': 167, 'start_index': 0}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ Z

In [75]:
len(document_ids)

321

👉 Bir vektör deposundaki belgenin içeriğine ve meta verilerine nasıl erişebilirsiniz?

In [76]:
one_doc = vector_store.get_by_ids(document_ids[:1])[0]
display(Markdown(one_doc.page_content))
display(one_doc.metadata)

2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV 
TUARASCÁIL FOCAL AR FHOCAL NA N-IMEACHTAÍ  DOBESEDNI ZAPISI RAZPRAV 
DOSLOVNO IZVJEŠĆE  SANATARKAT ISTUNTOSELOSTUKSET 
RESOCONTO INTEGRALE DELLE DISCUSSIONI  FULLSTÄNDIGT FÖRHANDLINGSREFERAT 
 
 
Сряда - Miércoles - Středa - Onsdag - Mittwoch - Kolmapäev - Τετάρτη - Wednesday 
Mercredi - Dé Céadaoin - Srijeda - Mercoledì - Trešdiena - Trečiadienis - Szerda - L-Erbgħa 
Woensdag - Środa - Quarta-feira - Miercuri - Streda - Sreda - Keskiviikko - Onsdag 
 
25.03.2026 
 
 
 
Единство в многообразието - Unida en la diversidad - Jednotná v rozmanitosti - Forenet i mangfoldighed - In Vielfalt geeint - Ühinenud mitmekesisuses 
Eνωμένη στην πολυμορφία - United in diversity - Unie dans la diversité - Aontaithe san éagsúlacht - Ujedinjena u raznolikosti - Unita nella diversità 
Vienoti daudzveidībā - Suvienijusi įvairovę - Egyesülve a sokféleségben - Magħquda fid-diversità - In verscheidenheid verenigd - Zjednoczona w różnorodności 
Unida na diversidade - Unită în diversitate - Zjednotení v rozmanitosti - Združena v raznolikosti - Moninaisuudessaan yhtenäinen - Förenade i mångfalden 
 
Редактирана версия - Edición revisada - Revidované vydání - Revideret udgave - Überprüfte Ausgabe - Uuendatud versioon 
Αναθεωρημένη έκδοση - Revised edition - Edition révisée - Eagrán athbhreithnithe - Revidirano izdanje - Edizione rivista

{'producer': 'Aspose.Words for Java 24.2.0',
 'creator': 'Aspose.Words',
 'creationdate': '',
 'author': 'e-Parliament@europarl.europa.eu',
 'dmxml.render.id': '440386',
 'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef',
 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml',
 'source': 'data/CRE-10-2026-03-25_EN.PDF',
 'total_pages': 167,
 'start_index': 0}

## 🔎 Benzer belgeleri almak için vektör deposunu kullan

Artık belgeleri gömleğe çevirdiğimize göre, benzer belgeleri almak için vektör deposunu kullanabiliriz.

👉 Bunun nasıl çalıştığını görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası üzerine tartışmayı özetle.", ve en benzer belgeleri bulun. Ayrıca alınacak belge sayısını da belirtebilirsiniz.

In [77]:
# Save your question into a variable called `query`

query = "Summarize the discussion on agricultural policy."

# Use the vector store to find similar documents to the query. Store the result in a variable called `retrieved_docs`

retrieved_docs = vector_store.similarity_search(query, k=6)

In [78]:
for doc in retrieved_docs:
    display(Markdown(doc.page_content))

européenne, voilà la signature, le 24 mars, dans la foulée, du traité de libre échange avec 
l'Australie. Comme dans le traité avec l'Amérique du Sud, c'est notre agriculture qui va payer une 
sévère note. 30 000 tonnes de viande bovine, 25 000 tonnes de viande ovine, 35 000 tonnes de 
sucre, de la poudre de lait, du beurre vont être importés. 
 
Méthodiquement, l'Union européenne organise la mort de notre agriculture. Le cumul de ces 
traités de libre échange, des contraintes imposées à nos agriculteurs et l'annonce d'un 
effondrement du budget futur de la PAC le confirment. 
 
Il y a quelques jours, la Commission européenne a reconnu que réduire la part du nucléaire était, je 
cite, ouvrez les guillemets, «une erreur stratégique». Il faudra encore combien de temps, combien 
d'années pour que l'Union européenne reconnaisse l'erreur majeure de nous faire perdre notre 
souveraineté alimentaire et donc d'aggraver notre dépendance? 
 
Méfiez-vous, la colère gronde dans les campagnes contre vos politiques mortifères. Nous vous 
aurons prévenus. 
 
1-0146-0000 
Carlo Fidanza (ECR). – Signora Presidente, signor Commissario, presidente Costa, onorevoli 
colleghi, l'Europa è sotto pressione su più fronti e non può più permettersi risposte lente né 
tantomeno ideologiche. 
 
La crisi in Medio Oriente ci impone realismo e rapidità di azione perché incide direttamente sul 
costo dell'energia, l'approvvigionamento di materie prime critiche, i traffici commerciali e, se 
dovesse perdurare, anche sulla sicurezza interna europea con possibili nuovi flussi migratori. Un 
rischio che dobbiamo prevenire se non vogliamo farci male, come fu con la folle gestione dei flussi 
dalla Siria nel 2015. 
 
E mentre si attende l'apertura concreta di uno spiraglio negoziale che possa portare alla fine del 
conflitto e alla rinuncia definitiva dell'Iran ai suoi piani nucleari, l'Unione europea dovrebbe fare la

européen donne ses permis pour changer le budget, personne ne peut arrêter une décision qui est 
en cours et qui manque seulement sa formalisation finale. Parce que le Parlement européen a 
donné son consentement dans les précis termes que le Conseil avait demandés. 
 
Et finalement, on doit agir de bonne foi. Et ce n'est pas agir de bonne foi quand on nous pose une 
condition qui ne dépend ni de quelques États membres, ni d'aucune institution, mais de la capacité 
de l'Ukraine à réparer le pipeline et la volonté de la Russie de le détruire une fois encore, ou de ne 
pas détruire une fois encore. Et il faut rappeler que la Russie a déjà attaqué 23 fois ce pipeline et 
qu'on ne peut pas être dépendants de nos décisions, des décisions de la Russie. Cela, on ne peut pas 
accepter. 
 
Pour notre compétitivité et pour notre croissance, pour les emplois des Européens, c'est essentiel, 
notre politique commerciale. Et aussi pour notre agriculture. Parce que par la politique 
commerciale, on ouvre de nouvelles, de nouveaux marchés pour nos fromages, pour nos vins,
88  25-03-2026 
pour nos productions agricoles. Et dans nos traités de commerce, évidemment, il nous faut 
protéger nos intérêts, notamment dans l'agriculture. C'est ce qu'on a fait dans le Mercosur. Parce 
qu'on parle du Mercosur, mais on oublie souvent que, quand on parle par exemple de bœuf, il y a 
un quota de 1,5 %. Quand on parle du sucre, il y a un autre quota de moins de 2 %. Quand on parle 
du lait, il y a un quota de moins de 2 %. 
 
Et c'est la même chose avec l'Australie, parce qu'avec l'Australie aussi, il y aura un quota. 
Finalement. 
 
We are not paralysed. In spite of the world instability, we keep our focus on our own agenda and 
it's what we have done last week in the European Council – because in spite of the crisis in Iran, or 
the war in Ukraine, we keep our focus to launch 'one Europe, one market' agenda with concrete

Víctor Torres, a la Estrategia para las Regiones Ultraperiféricas 2027-2030 con una comunicación 
muy concreta.
144  25-03-2026 
Esta comunicación fue, en primer lugar, apoyo inequívoco al programa de opciones específicas del 
sector primario del POSEI; en segundo lugar, solidaridad en el Pacto de Migraciones, en particular 
en las regiones exteriores y apoyo a los realojamientos en los rescates y salvamento en la mar; y, en 
tercer lugar, una opción específica de intervención del mercado de la vivienda para abaratar los 
costes de la vivienda en lo relacionado con la compra por fondos buitre y por fondos 
especulativos. 
 
1-0329-0000 
Valérie Deloge (PfE). – Madame la Présidente, Mesdames et Messieurs de la Commission 
européenne, vous avez signé un nouvel accord de libre-échange avec l'Australie dans l'opacité et la 
précipitation, au mépris des alertes des agriculteurs. Vos négociations avec l'Australie prévoient 
l'entrée de plus de 30 000 tonnes de bœuf et autant de tonnes d'agneau en moyenne. Une fois de 
plus, l'agriculture sert de variable d'ajustement. 
 
Votre scénario, nous le connaissons déjà, c'est le même que le Mercosur: promesses de contrôles, 
discours rassurants, mais, dans les faits, une réalité bien différente. Avec le Mercosur, les faits sont 
accablants, la traçabilité est défaillante. Des contrôles révèlent que des animaux traités à 
l'œstradiol 17β, hormone interdite en Europe, ont été déclarés éligibles à l'exportation. Ces 
produits ont été envoyés vers l'Europe sans que les importateurs en soient informés. Et voilà donc 
des produits dangereux dans nos assiettes. 
 
Et pourtant, la Commission persiste. Allons-nous assister à un nouveau coup de force 
antidémocratique, une application sans vote des députés européens et des parlements nationaux? 
Nous avons alerté sur le Mercosur. Les faits nous donnent raison. Vous êtes responsables et 
coupables. Il faut arrêter cette fuite en avant. 
 
1-0330-0000

Face au risque d'aggraver la concurrence directe contre nos paysans, déjà écrasés de normes et 
d'obsessions décroissantistes, nous souhaitons qu'un débat puisse se tenir dans ce Parlement. Dans 
un contexte international qui nous rappelle la nécessité existentielle d'être souverains sur le plan 
alimentaire, il va de soi que ce nouvel accord constituerait une énième folie et une trahison envers 
ceux qui nous nourrissent. Je vous propose donc l'inscription, à l'ordre du jour de notre session, du 
débat au titre suivant: «Accord de libre-échange Union européenne-Australie: conséquences sur 
l'agriculture européenne». 
 
De grâce, mes chers collègues, réveillez-vous et ne menons pas nos agriculteurs à l'abattoir.
32  25-03-2026 
1-0026-0000 
President. – I have received an alternative proposal from The Left Group, which would have the 
title as follows: Commission statement on 'EU-Australia Free Trade Agreement and its impacts on 
agriculture, health and the environment'. Mr Bardella, do you agree with the alternative proposal 
of The Left Group? 
 
You agree. OK. So we will vote on The Left Group proposal by roll call. 
 
(Parliament rejected the request) 
 
The next proposal is a request by the ECR Group that Council and Commission statements on 
'The need to combat antisemitism and protect Jewish life in Europe, following the recent terrorist 
attacks against the Jewish community in the Netherlands and Belgium' be added as the fourth item 
in the afternoon. 
 
As a consequence, the sitting would be extended to 23:00. 
 
I give the floor to Bert-Jan Ruissen to move the request on behalf of the ECR Group. 
 
1-0027-0000 
Bert-Jan Ruissen, namens de ECR-Fractie. – Voorzitter, het houdt maar niet op. En het gaat van 
kwaad naar erger. Ik bedoel de lange reeks van aanslagen op de joodse gemeenschap – de 
Voorzitter noemde ze al – zoals recent op een synagoge in Luik, een synagoge in Rotterdam, een

sowie der Vatikan verurteilen die wachsende Gewalt gegen unsere Glaubensbrüder ebenso wie 
Israels ehemaliger Premier Naftali Bennett. Der Schutz der Christen im Nahen Osten muss 
unverhandelbare Konditionalität in unserer Außen- und Entwicklungspolitik werden. Wer 
Christen verfolgt, darf nicht von Brüssel finanziert und hofiert werden. 
 
1-0341-0000 
Ruth Firmenich (NI). – Frau Präsidentin! Die US-Politik gegenüber Kuba ist 
menschenverachtend. Kuba soll ausgehungert werden, um so einen Regimewechsel zu erreichen. 
Was 65 Jahre US-Wirtschaftsembargo nicht geschafft haben, will Präsident Trump jetzt durch das 
vollständige Abschneiden Kubas von Energielieferungen erreichen. Es gab schon mehrfach einen 
landesweiten Stromausfall. Das bedeutet nicht nur, dass öffentliches Leben, Verkehr und 
Produktion zum Erliegen kommen. Es bedeutet auch, dass Kühlketten nicht mehr funktionieren, 
Krankenhäuser die Patienten nicht mehr versorgen können, Impfstoffe verrotten. 
 
10 Millionen Menschen leben in Kuba. Die USA nehmen diese gezielt ins Visier, um über ihr Leid 
das System zu stürzen. Und die EU schaut tatenlos zu. Doch wer schweigt, macht sich 
mitschuldig. Das Aushungern eines Landes ist völkerrechtswidrig. Aber Menschenrechte und 
Völkerrecht interessieren die EU nur, wenn es passt. Diese Doppelmoral muss ein Ende haben. 
Kuba braucht unsere Solidarität. Die EU muss dafür sorgen, dass Kuba Unterstützung erhält. 
 
1-0342-0000 
Diana Iovanovici Şoşoacă (NI). – Doamnă președintă, ați discriminat agricultorii români și ați 
distrus toată agricultura românească, ați distrus creșterea animalelor, ne-ați vândut, ne-ați luat 
resursele, dar considerați că toți membrii Uniunii Europene sunt egali? Nu, nu sunt egali. 
 
Este o ipocrizie și o dublă măsură ce există în Uniunea Europeană. Măsurile de acum arată că 
Uniunea Europeană trebuie să se desființeze. Cea mai mare parte din români cer ieșirea din 
Uniunea Europeană, iar eu voi spune aici adevărul: Roexit!

with the adaptation of this file. 
 
Our simplification agenda continues apace. The European Council called on the Commission, the 
co-legislators and the Member States to continue to ambitiously simplify rules and reduce 
administrative burdens at the EU, national and regional level. We have set an ambitious target of 
25 % cut in administrative burdens for all businesses and 35 % for SMEs. 
 
Last year, the Commission tabled 10 omnibus proposals. These proposed measures have potential 
for EUR 15 billion reduction of recurring administrative costs and EUR 6 billion in one-off 
savings. These are real savings that contribute to boosting the competitiveness of EU companies. 
The European Council called on the co-legislators to agree all pending omnibus proposals before 
the end of 2026. 
 
This year, more than half of the legislative initiatives in the Commission's work programme have a 
strong simplification focus. This means that they must deliver net administrative savings. In 
addition to further omnibus proposals, the Commission will introduce 'simplicity by design' 
approach to future rules so that we improve the governance and development of legal texts. 
 
Finally, in the broader context of a further deteriorating geopolitical environment and Russia's war 
of aggression against Ukraine remaining an existential challenge for the European Union, the 
European Council also reaffirmed its determination to decisively ramp up Europe's defence 
readiness by 2030.
68  25-03-2026 
 
The leaders also took stock of the progress on migration and held a lunch with UN Secretary-
General Guterres. 
 
And finally, leaders decided that the discussions on the next multiannual financial framework 
would be postponed to the informal meeting of heads of state and government on 23 and 
24 April in Cyprus. 
 
To conclude, honourable Members, we must continue to act with determination and focus to

Bu, RAG'ın sözde "Alma" (Retrieval) kısmını tamamlar: artık sorgumuza en benzer belgeleri bulabiliriz.

Çalışmanın çoğu artık tamamlandı!

## 💬 Sorumuza bir cevap üret

Şimdiye kadar benzer belgeleri almamızı sağlamak için sadece bir **gömme modeli** kullandık.

Şimdi, sorumuzla bir cevap almak için üretici bir LLM kullanacağız: ona aldığımız belgeler ve sorumuzla besleyeceğiz.

Bunu yapmanın en temel yolu tüm girdilerimizi birbirine bağlamak, sorumuzla eklemek ve sonucu görmek olacaktır.

Bir deneyelim.

👉 İlk olarak önceki meydan okumalarda olduğu gibi bir LLM başlatın.

In [81]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

Sonra temel bir istem oluşturun:

In [82]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi istemi kullanın:

In [83]:
response = llm.invoke(prompt)
Markdown(response.content)

The provided texts reveal a significant debate and concern within the European Union regarding agricultural policy, particularly in relation to free trade agreements. Here's a summary of the key points discussed:

**Core Criticisms and Concerns:**

*   **Negative Impact of Free Trade Agreements:** Several speakers express strong opposition to recent and proposed free trade agreements, specifically mentioning those with Australia and South America (Mercosur). The primary concern is that these agreements will severely harm European agriculture.
*   **Agricultural Sacrifice:** Agriculture is repeatedly described as being used as a "severe note" or a "variable d'ajustement" in these trade deals. This means that concessions made in agricultural imports are seen as the cost of securing broader trade benefits.
*   **Increased Imports and Competition:** Specific quantities of agricultural products like beef, lamb, sugar, milk powder, and butter are mentioned as being allowed to be imported under these agreements. This is seen as direct competition that will crush European farmers, who are already burdened by regulations and what some call "decroissantist obsessions."
*   **Loss of Food Sovereignty and Increased Dependence:** A major concern is that these policies are leading to a loss of food sovereignty for the EU and increasing its dependence on external suppliers. This is directly linked to the idea that the EU is "organizing the death of our agriculture."
*   **Failure to Learn from Past Mistakes:** The analogy is drawn to the European Commission recognizing a "strategic error" in reducing nuclear power. Speakers question how long it will take for the EU to acknowledge the "major error" of undermining food sovereignty.
*   **Lack of Transparency and Democratic Process:** Concerns are raised about the "opaqueness and precipitation" with which some agreements, like the one with Australia, were signed, and the potential for their application without a vote from MEPs or national parliaments.
*   **Doubts about Controls and Traceability:** Following the Mercosur agreement, there are accusations that controls are failing, and products treated with banned hormones (like oestradiol 17β) have been exported to the EU without proper information for importers, leading to potentially dangerous products on consumers' plates.
*   **Broader Policy Context:** The cumulative effect of these trade deals, combined with constraints on farmers and announced future budget cuts to the Common Agricultural Policy (CAP), is seen as a deliberate strategy to weaken European agriculture.

**Arguments in Defense (or Nuance):**

*   **Protection of Interests within Treaties:** One speaker (presumably from the Commission or a supporter of the agreements) argues that the EU *does* protect its interests in trade treaties. They highlight that even in the Mercosur agreement, there are quotas for beef (1.5%), sugar (less than 2%), and milk (less than 2%). Similar quotas are expected for Australia.
*   **Opening New Markets:** The same speaker emphasizes that commercial policy is essential for competitiveness and growth, opening new markets for European products like cheeses and wines, as well as other agricultural productions.
*   **Focus on Agenda Despite Instability:** Despite global instability (crises in Iran, war in Ukraine), the EU is maintaining its focus on its own agenda, including trade initiatives.

**Calls to Action and Warnings:**

*   **Demand for Debate:** Several speakers call for a debate in Parliament on the specific consequences of the EU-Australia Free Trade Agreement on European agriculture.
*   **Warning of Anger:** There is a clear warning that "anger is growing in the countryside against your deadly policies."
*   **Call to Stop the "Rush Forward":** There is an urgent plea to stop what is perceived as a reckless and detrimental course of action.
*   **"Roexit" (Romanian Exit from EU):** One Romanian MEP expresses extreme dissatisfaction, stating that the EU has discriminated against Romanian farmers and that the majority of Romanians want to leave the EU.

**Overall Sentiment:**

The dominant sentiment among a significant portion of the speakers is one of **anger, frustration, and deep concern** regarding the impact of the EU's trade policies on its agricultural sector. They perceive a pattern of sacrificing farmers and food sovereignty for broader economic or political goals, with a lack of transparency and a disregard for the warnings from those on the ground.

Bu fena değil, ama modele daha fazla rehberlik vererek daha kapsamlı bir istem yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değilmişiz ve LangChain'in bizim için önceden hazırlanmış istem kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu görmezden gelebilirsiniz.)

In [84]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)



You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


LangChain'in bizim için nasıl daha kesin bir istem oluşturduğunu görüyor musunuz? Bunu RAG'ımız için kullanalım!

👉 İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.

In [85]:
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
Markdown(docs_content)

européenne, voilà la signature, le 24 mars, dans la foulée, du traité de libre échange avec 
l'Australie. Comme dans le traité avec l'Amérique du Sud, c'est notre agriculture qui va payer une 
sévère note. 30 000 tonnes de viande bovine, 25 000 tonnes de viande ovine, 35 000 tonnes de 
sucre, de la poudre de lait, du beurre vont être importés. 
 
Méthodiquement, l'Union européenne organise la mort de notre agriculture. Le cumul de ces 
traités de libre échange, des contraintes imposées à nos agriculteurs et l'annonce d'un 
effondrement du budget futur de la PAC le confirment. 
 
Il y a quelques jours, la Commission européenne a reconnu que réduire la part du nucléaire était, je 
cite, ouvrez les guillemets, «une erreur stratégique». Il faudra encore combien de temps, combien 
d'années pour que l'Union européenne reconnaisse l'erreur majeure de nous faire perdre notre 
souveraineté alimentaire et donc d'aggraver notre dépendance? 
 
Méfiez-vous, la colère gronde dans les campagnes contre vos politiques mortifères. Nous vous 
aurons prévenus. 
 
1-0146-0000 
Carlo Fidanza (ECR). – Signora Presidente, signor Commissario, presidente Costa, onorevoli 
colleghi, l'Europa è sotto pressione su più fronti e non può più permettersi risposte lente né 
tantomeno ideologiche. 
 
La crisi in Medio Oriente ci impone realismo e rapidità di azione perché incide direttamente sul 
costo dell'energia, l'approvvigionamento di materie prime critiche, i traffici commerciali e, se 
dovesse perdurare, anche sulla sicurezza interna europea con possibili nuovi flussi migratori. Un 
rischio che dobbiamo prevenire se non vogliamo farci male, come fu con la folle gestione dei flussi 
dalla Siria nel 2015. 
 
E mentre si attende l'apertura concreta di uno spiraglio negoziale che possa portare alla fine del 
conflitto e alla rinuncia definitiva dell'Iran ai suoi piani nucleari, l'Unione europea dovrebbe fare la

européen donne ses permis pour changer le budget, personne ne peut arrêter une décision qui est 
en cours et qui manque seulement sa formalisation finale. Parce que le Parlement européen a 
donné son consentement dans les précis termes que le Conseil avait demandés. 
 
Et finalement, on doit agir de bonne foi. Et ce n'est pas agir de bonne foi quand on nous pose une 
condition qui ne dépend ni de quelques États membres, ni d'aucune institution, mais de la capacité 
de l'Ukraine à réparer le pipeline et la volonté de la Russie de le détruire une fois encore, ou de ne 
pas détruire une fois encore. Et il faut rappeler que la Russie a déjà attaqué 23 fois ce pipeline et 
qu'on ne peut pas être dépendants de nos décisions, des décisions de la Russie. Cela, on ne peut pas 
accepter. 
 
Pour notre compétitivité et pour notre croissance, pour les emplois des Européens, c'est essentiel, 
notre politique commerciale. Et aussi pour notre agriculture. Parce que par la politique 
commerciale, on ouvre de nouvelles, de nouveaux marchés pour nos fromages, pour nos vins,
88  25-03-2026 
pour nos productions agricoles. Et dans nos traités de commerce, évidemment, il nous faut 
protéger nos intérêts, notamment dans l'agriculture. C'est ce qu'on a fait dans le Mercosur. Parce 
qu'on parle du Mercosur, mais on oublie souvent que, quand on parle par exemple de bœuf, il y a 
un quota de 1,5 %. Quand on parle du sucre, il y a un autre quota de moins de 2 %. Quand on parle 
du lait, il y a un quota de moins de 2 %. 
 
Et c'est la même chose avec l'Australie, parce qu'avec l'Australie aussi, il y aura un quota. 
Finalement. 
 
We are not paralysed. In spite of the world instability, we keep our focus on our own agenda and 
it's what we have done last week in the European Council – because in spite of the crisis in Iran, or 
the war in Ukraine, we keep our focus to launch 'one Europe, one market' agenda with concrete

Víctor Torres, a la Estrategia para las Regiones Ultraperiféricas 2027-2030 con una comunicación 
muy concreta.
144  25-03-2026 
Esta comunicación fue, en primer lugar, apoyo inequívoco al programa de opciones específicas del 
sector primario del POSEI; en segundo lugar, solidaridad en el Pacto de Migraciones, en particular 
en las regiones exteriores y apoyo a los realojamientos en los rescates y salvamento en la mar; y, en 
tercer lugar, una opción específica de intervención del mercado de la vivienda para abaratar los 
costes de la vivienda en lo relacionado con la compra por fondos buitre y por fondos 
especulativos. 
 
1-0329-0000 
Valérie Deloge (PfE). – Madame la Présidente, Mesdames et Messieurs de la Commission 
européenne, vous avez signé un nouvel accord de libre-échange avec l'Australie dans l'opacité et la 
précipitation, au mépris des alertes des agriculteurs. Vos négociations avec l'Australie prévoient 
l'entrée de plus de 30 000 tonnes de bœuf et autant de tonnes d'agneau en moyenne. Une fois de 
plus, l'agriculture sert de variable d'ajustement. 
 
Votre scénario, nous le connaissons déjà, c'est le même que le Mercosur: promesses de contrôles, 
discours rassurants, mais, dans les faits, une réalité bien différente. Avec le Mercosur, les faits sont 
accablants, la traçabilité est défaillante. Des contrôles révèlent que des animaux traités à 
l'œstradiol 17β, hormone interdite en Europe, ont été déclarés éligibles à l'exportation. Ces 
produits ont été envoyés vers l'Europe sans que les importateurs en soient informés. Et voilà donc 
des produits dangereux dans nos assiettes. 
 
Et pourtant, la Commission persiste. Allons-nous assister à un nouveau coup de force 
antidémocratique, une application sans vote des députés européens et des parlements nationaux? 
Nous avons alerté sur le Mercosur. Les faits nous donnent raison. Vous êtes responsables et 
coupables. Il faut arrêter cette fuite en avant. 
 
1-0330-0000

Face au risque d'aggraver la concurrence directe contre nos paysans, déjà écrasés de normes et 
d'obsessions décroissantistes, nous souhaitons qu'un débat puisse se tenir dans ce Parlement. Dans 
un contexte international qui nous rappelle la nécessité existentielle d'être souverains sur le plan 
alimentaire, il va de soi que ce nouvel accord constituerait une énième folie et une trahison envers 
ceux qui nous nourrissent. Je vous propose donc l'inscription, à l'ordre du jour de notre session, du 
débat au titre suivant: «Accord de libre-échange Union européenne-Australie: conséquences sur 
l'agriculture européenne». 
 
De grâce, mes chers collègues, réveillez-vous et ne menons pas nos agriculteurs à l'abattoir.
32  25-03-2026 
1-0026-0000 
President. – I have received an alternative proposal from The Left Group, which would have the 
title as follows: Commission statement on 'EU-Australia Free Trade Agreement and its impacts on 
agriculture, health and the environment'. Mr Bardella, do you agree with the alternative proposal 
of The Left Group? 
 
You agree. OK. So we will vote on The Left Group proposal by roll call. 
 
(Parliament rejected the request) 
 
The next proposal is a request by the ECR Group that Council and Commission statements on 
'The need to combat antisemitism and protect Jewish life in Europe, following the recent terrorist 
attacks against the Jewish community in the Netherlands and Belgium' be added as the fourth item 
in the afternoon. 
 
As a consequence, the sitting would be extended to 23:00. 
 
I give the floor to Bert-Jan Ruissen to move the request on behalf of the ECR Group. 
 
1-0027-0000 
Bert-Jan Ruissen, namens de ECR-Fractie. – Voorzitter, het houdt maar niet op. En het gaat van 
kwaad naar erger. Ik bedoel de lange reeks van aanslagen op de joodse gemeenschap – de 
Voorzitter noemde ze al – zoals recent op een synagoge in Luik, een synagoge in Rotterdam, een

sowie der Vatikan verurteilen die wachsende Gewalt gegen unsere Glaubensbrüder ebenso wie 
Israels ehemaliger Premier Naftali Bennett. Der Schutz der Christen im Nahen Osten muss 
unverhandelbare Konditionalität in unserer Außen- und Entwicklungspolitik werden. Wer 
Christen verfolgt, darf nicht von Brüssel finanziert und hofiert werden. 
 
1-0341-0000 
Ruth Firmenich (NI). – Frau Präsidentin! Die US-Politik gegenüber Kuba ist 
menschenverachtend. Kuba soll ausgehungert werden, um so einen Regimewechsel zu erreichen. 
Was 65 Jahre US-Wirtschaftsembargo nicht geschafft haben, will Präsident Trump jetzt durch das 
vollständige Abschneiden Kubas von Energielieferungen erreichen. Es gab schon mehrfach einen 
landesweiten Stromausfall. Das bedeutet nicht nur, dass öffentliches Leben, Verkehr und 
Produktion zum Erliegen kommen. Es bedeutet auch, dass Kühlketten nicht mehr funktionieren, 
Krankenhäuser die Patienten nicht mehr versorgen können, Impfstoffe verrotten. 
 
10 Millionen Menschen leben in Kuba. Die USA nehmen diese gezielt ins Visier, um über ihr Leid 
das System zu stürzen. Und die EU schaut tatenlos zu. Doch wer schweigt, macht sich 
mitschuldig. Das Aushungern eines Landes ist völkerrechtswidrig. Aber Menschenrechte und 
Völkerrecht interessieren die EU nur, wenn es passt. Diese Doppelmoral muss ein Ende haben. 
Kuba braucht unsere Solidarität. Die EU muss dafür sorgen, dass Kuba Unterstützung erhält. 
 
1-0342-0000 
Diana Iovanovici Şoşoacă (NI). – Doamnă președintă, ați discriminat agricultorii români și ați 
distrus toată agricultura românească, ați distrus creșterea animalelor, ne-ați vândut, ne-ați luat 
resursele, dar considerați că toți membrii Uniunii Europene sunt egali? Nu, nu sunt egali. 
 
Este o ipocrizie și o dublă măsură ce există în Uniunea Europeană. Măsurile de acum arată că 
Uniunea Europeană trebuie să se desființeze. Cea mai mare parte din români cer ieșirea din 
Uniunea Europeană, iar eu voi spune aici adevărul: Roexit!

with the adaptation of this file. 
 
Our simplification agenda continues apace. The European Council called on the Commission, the 
co-legislators and the Member States to continue to ambitiously simplify rules and reduce 
administrative burdens at the EU, national and regional level. We have set an ambitious target of 
25 % cut in administrative burdens for all businesses and 35 % for SMEs. 
 
Last year, the Commission tabled 10 omnibus proposals. These proposed measures have potential 
for EUR 15 billion reduction of recurring administrative costs and EUR 6 billion in one-off 
savings. These are real savings that contribute to boosting the competitiveness of EU companies. 
The European Council called on the co-legislators to agree all pending omnibus proposals before 
the end of 2026. 
 
This year, more than half of the legislative initiatives in the Commission's work programme have a 
strong simplification focus. This means that they must deliver net administrative savings. In 
addition to further omnibus proposals, the Commission will introduce 'simplicity by design' 
approach to future rules so that we improve the governance and development of legal texts. 
 
Finally, in the broader context of a further deteriorating geopolitical environment and Russia's war 
of aggression against Ukraine remaining an existential challenge for the European Union, the 
European Council also reaffirmed its determination to decisively ramp up Europe's defence 
readiness by 2030.
68  25-03-2026 
 
The leaders also took stock of the progress on migration and held a lunch with UN Secretary-
General Guterres. 
 
And finally, leaders decided that the discussions on the next multiannual financial framework 
would be postponed to the informal meeting of heads of state and government on 23 and 
24 April in Cyprus. 
 
To conclude, honourable Members, we must continue to act with determination and focus to

👉 Sonra, sorgunuz ve alınan belgelerden başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [86]:
prompt = prompt_template.invoke(
    {"context": docs_content, "question": query}
)

👉 Son olarak az önce oluşturduğumuz `the_prompt` ile LLM modelini kullanın:

In [87]:
answer = llm.invoke(prompt)

In [88]:
Markdown(answer.content)

The discussion on agricultural policy centers on the negative impacts of free trade agreements, such as the one with Australia and Mercosur, on European agriculture. Critics argue these agreements lead to significant import quotas of agricultural products like beef, lamb, sugar, and dairy, effectively harming domestic farmers and threatening food sovereignty. Concerns are also raised about the potential for the Common Agricultural Policy (CAP) budget to decrease, exacerbating these issues.

🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız belgelerde ***temelli*** metin üretti.

## 💾 Gömmelerimizi kalıcı hale getir

Şimdiye kadar bellekte vektör deposuyla çalıştık. Bu yüzden not defterinizi kapattığınızda, tüm gömmeleri de kaybedeceksiniz.

⚠️ Bu gömmelerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bedava çalışmazlar. 💰

Bunun gibi bir, nispeten küçük belge için maliyet düşüktür, ama hızla artar. Şimdiye kadar sadece bir günün transkriptleriyle çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birden fazla yıl var...

Bunu çözmek için sadece vektör depomuzla kalıcı bir taneyi değiştireceğiz. Bu LangChain'in avantajıdır: bileşenleri değiştirmek çok kolay.

Bellekteki vektör depomuz deneme için harikaydı, şimdi başka bir taneyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Bunu yerel olarak çalıştırabilir ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Her şeyi birkaç kod hücresinde tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştüreceğiz.

Sonunda iki fonksiyon istiyoruz:

1. `embed_and_store()`: Başka bir oturumun transkriptini vektör veritabanımıza ekle, böylece alacağımız daha fazla veri olsun.
2. `answer()`: Vektör depomuzla farklı sorularla sorgula.

#### 1. Bir Chroma vektör deposu başlat

👉 **Veri kalıcılığıyla** (yani verileri diskteki bir dizinde saklayarak) Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in belgelerine](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [89]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="ep_plenary",
    embedding_function=embeddings,
    persist_directory="./chroma_ep_follower",
)

#### 2. `embed_and_store()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [90]:
def embed_and_store(file_path, vector_store):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store."""
    # Load the PDF file
    loader = PyPDFLoader(file_path, mode='single')
    pdf = loader.load()

    # Split the pages into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2_000,  # chunk size (characters)
        chunk_overlap=400,  # chunk overlap (characters)
        add_start_index=True,  # track index in original document
    )
    all_splits = text_splitter.split_documents(pdf)

    # # Add the session_date to the metadata
    # for split in all_splits:
    #     split.metadata['session_date'] = session_date

    # Add the chunks to the vector store
    document_ids = vector_store.add_documents(documents=all_splits)
    print(f"Added {len(document_ids)} documents to the vector store.")

    return document_ids

👉 Fonksiyonunuzu bir dosya veya hatta iki dosyayla deneyin:

In [92]:
file_path = "data/CRE-10-2026-03-25_EN.pdf"

document_ids = embed_and_store(file_path, vector_store)

Added 321 documents to the vector store.


In [93]:
vector_store.get_by_ids(document_ids[:3])

[Document(id='6c8ba520-e55b-47d3-b222-2359398b5fae', metadata={'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef', 'creationdate': '', 'start_index': 0, 'creator': 'Aspose.Words', 'total_pages': 167, 'source': 'data/CRE-10-2026-03-25_EN.pdf', 'dmxml.render.id': '440386', 'producer': 'Aspose.Words for Java 24.2.0', 'author': 'e-Parliament@europarl.europa.eu', 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml'}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ Z

#### 3. `answer()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [94]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    retrieved_docs = vector_store.similarity_search(query, k=6)

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.content

👉 Fonksiyonunuzu beğendiğiniz bir sorguyla deneyin:

In [95]:
query = "What is being said about international trade?"
Markdown(answer(query, vector_store, llm, prompt_template))

International trade is discussed in relation to the EU-Australia Security and Defence Partnership and Trade Agreement, with concerns raised about the impact on agriculture, including imports of beef, lamb, sugar, milk powder, and butter. There is also a mention of trade agreements opening new markets for products like cheese and wine, and the need to protect national interests within these agreements.

🏁 Tebrikler! Artık LangChain kullanarak RAG'da ustalaştınız ve vektör deponuza daha fazla belge eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar yapmayı öğrendiniz.

## [İsteğe Bağlı] Meta veri ekleme

Kurduğumuz RAG, vektör deposundaki tüm belgeleri sorgular. Orada birden fazla yılın bilgisinin olduğunu düşünün. Yıllara veya tarihlere göre filtreyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparız? Vektör deposundaki belgelerin meta veri içerdiğini unutmayın. Eğer tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabilirdik.

İpucu: Meta verilerinizi pipeline'ınızda olabildiğince erken ekleyin. Verileriniz vektör deposunda saklandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [ ]:
def embed_and_store_fancy(file_path, vector_store, session_date):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store.
    Session_date is added to the metadata of each chunk."""
    pass  # YOUR CODE HERE

    return document_ids

👉 Fonksiyonunuzu deneyin ve vektör deponuzun ek meta veri içerdiğini kontrol edin.

In [ ]:
# YOUR CODE HERE

Şimdi alıcıyı kullanıcının sorduğu tarihe göre sınırlamamız gerekiyor.

👉 `answer()` fonksiyonunuzu bir tarih alabilecek ve yeni meta verilere dayalı olarak belgeleri filtreleyebilecek şekilde uyarlayın.

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını meta veri aramasıyla birleştirdiniz!